# Illustration of anisotropic CBO an other methods on Rastrigin

This notebook compares teh behavior of CBO and its varaints on Rastrigin and a rotated variant.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('./..')
from cbx.dynamics import CBO, CBS
from cbx.scheduler import multiply
from cbx.utils.termination import max_eval_term
from cbx.noise import covariance_noise
import numpy as np
import matplotlib.pyplot as plt
from experimentUtils import check_unit_weight_estimates
from CBO_utils import PreconditionedFunction, RotFunCBO, RotNoiseCBO, SplitDimCBO, Rot_fun, estimate_Q
from problems import RastriginProblem

## Set up the problem

In [ ]:
np.random.seed(211)
d   = 2
P   = RastriginProblem(d=d)
f   = P.generate_transformed(quasi_orth=True, perturb = 0.)['objective']
N   = 10
x0  = P.sample(1, N)

## Run CBO on standard Rastrigin function

In [ ]:
ckwargs = {
    'dt':0.05, 'sigma':5., 'lamda':1., 'max_it':80,
    'track_args': {'names':['x', 'consensus', 'energy']},
    'alpha':1e2, 'verbosity':0, 'seed': 42
}
print_summary = lambda dyn: print('Finished with energy ', dyn.f(dyn.best_particle))
dyn = CBO(f.f,x=x0.copy(), noise = 'anisotropic', **ckwargs)
dyn.optimize()

print_summary(dyn)

## Run CBO on rotated Rastrigin

In [ ]:
dyn_f_rot = CBO(f,x=x0.copy(), noise = 'anisotropic', **ckwargs)
dyn_f_rot.optimize()

print_summary(dyn_f_rot)

## Run CBO on rotated Rastrigin with isotropic noise

In [ ]:
dyn_iso = CBO(f, x=x0.copy(), noise = 'isotropic', **(ckwargs | {'sigma':2.}))
dyn_iso.optimize()
print_summary(dyn_iso)

## Plot results

In [ ]:
from cbx.plotting import PlotDynamicHistory
import scienceplots
plt.style.use('science')
fig, ax = plt.subplots(1,1, figsize=(6,6))

rot, noise = True, 'aniso'
dyn_plot = dyn_iso if noise == 'iso' else (dyn_rot_CBS if noise == 'CBS' else dyn_f_rot if rot else dyn)
pl = PlotDynamicHistory(
    dyn_plot, objective_args={'x_min':-4, 'x_max':4, 'cmap':'cividis','levels':150, 'num_pts':200},
    ax=ax)
pl.plot_objective()
x = np.array(dyn_plot.history['x'])
ts = np.arange(30, x.shape[0], 3)
ps = np.arange(30, x.shape[0], 3)
for i in range(N): 
    plt.plot(x[ps,0,i,0], x[ps,0,i,1], color='w', alpha=0.3)
    plt.scatter(x[ts,0,i,0], x[ts,0,i,1], color='w', alpha=0.5, s=15, label='Particles' if i == 0 else None)

glob_min = P.minimum if rot else f.f.b
plt.scatter(glob_min[0], glob_min[1], color='red', label='Global minimum', marker='$\odot$', s=100)
plt.legend(frameon=True, facecolor='xkcd:sky')
plt.savefig(f'../results/figs/CBO_2d_{"rot_" if rot else "" }{noise}.png')

## Estimate matrix

In [ ]:
Qest_kwargs = {'n_hess':100, 'sval_cutoff':0.5, 'eps_hessian':1e-5, 'restarts' : 1600}
Q = estimate_Q(f, d = d, **Qest_kwargs)
print("Number of true weights | wrong estimates | missed weights")
check_unit_weight_estimates(f.Q, Q, just_counts=True, prec = 1e-6)

## Noise rotation

In [ ]:
dyn_RN = RotNoiseCBO(f, x=x0.copy(), Q = Q, noise = 'anisotropic', **ckwargs)
dyn_RN.optimize()
print_summary(dyn_RN)

## Plot result

In [ ]:
from cbx.plotting import PlotDynamicHistory
import scienceplots
plt.style.use('science')
fig, ax = plt.subplots(1,1, figsize=(6,6))

dyn_plot = dyn_RN
pl = PlotDynamicHistory(
    dyn_plot, objective_args={'x_min':-4, 'x_max':4, 'cmap':'cividis','levels':150, 'num_pts':200},
    ax=ax)
pl.plot_objective()
x = np.array(dyn_plot.history['x'])
ts = np.arange(30, x.shape[0], 3)
ps = np.arange(30, x.shape[0], 3)
for i in range(N): 
    plt.plot(x[ps,0,i,0], x[ps,0,i,1], color='w', alpha=0.3)
    plt.scatter(x[ts,0,i,0], x[ts,0,i,1], color='w', alpha=0.5, s=15, label='Particles' if i == 0 else None)

glob_min = P.minimum
plt.scatter(glob_min[0], glob_min[1], color='red', label='Global minimum', marker='$\odot$', s=100)
plt.legend(frameon=True, facecolor='xkcd:sky')
plt.savefig(f'figs/CBO_2d_rotnoise.png')